# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [4]:
# Import the project dependencies
import json
import os

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [5]:
# Load environment variables from starter/.env
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print(f"OPENAI_API_KEY loaded: {bool(OPENAI_API_KEY)}")
print(f"TAVILY_API_KEY loaded: {bool(TAVILY_API_KEY)}")

OPENAI_API_KEY loaded: True
TAVILY_API_KEY loaded: True


In [10]:
retrieval_results = retrieve_game("When was Pokémon Gold and Silver released?")
print(f"Retrieved {len(retrieval_results)} documents")
print(retrieval_results[0] if retrieval_results else "No results")

Retrieved 3 documents
{'document': '[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'metadata': {'Name': 'Pokémon Gold and Silver', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'Genre': 'Role-playing', 'YearOfRelease': 1999, 'Platform': 'Game Boy Color', 'Publisher': 'Nintendo'}, 'distance': 0.11277824640274048}


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# Connect to the persistent collection created in Part 1.
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY
)
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

@tool
def retrieve_game(query: str) -> list[dict]:
    """Search the game knowledge base for games relevant to a question."""
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    return [
        {
            "document": document,
            "metadata": metadata,
            "distance": distance,
        }
        for document, metadata, distance in zip(documents, metadatas, distances)
    ]

#### Evaluate Retrieval Tool

In [7]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents answer the question")
    description: str = Field(description="Why the documents are or are not sufficient")


evaluation_llm = LLM(model="gpt-4o-mini", temperature=0)

@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: list[dict],
) -> dict:
    """Assess whether retrieved game documents are sufficient to answer a question."""
    prompt = (
        "Evaluate whether the retrieved documents are sufficient to answer the question. "
        "Return useful=true only when the documents contain the facts needed for a reliable answer. "
        "Explain the decision briefly.\n\n"
        f"Question: {question}\n"
        f"Retrieved documents: {json.dumps(retrieved_docs, ensure_ascii=True)}"
    )
    response = evaluation_llm.invoke(prompt, response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(response.content).model_dump()


#### Game Web Search Tool

In [8]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> list[dict]:
    """Search the web for current or missing video game information."""
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
    )
    return response.get("results", [])

### Agent

In [9]:
agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0,
    instructions=(
        "You are UdaPlay, a careful video game research assistant. "
        "For game facts, call retrieve_game first. Then call evaluate_retrieval with the "
        "original question and retrieved documents. If the evaluation says the documents "
        "are not useful, call game_web_search. Answer concisely and distinguish facts from "
        "uncertainty. Never invent release dates or platforms."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

In [11]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for question in questions:
    run = agent.invoke(question)
    final_state = run.get_final_state()
    print(f"Question: {question}")
    print(f"Answer: {final_state['messages'][-1].content}")
    print(f"Tokens: {final_state.get('total_tokens', 0)}")
    print()

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Question: When was Pokémon Gold and Silver released?
Answer: Pokémon Gold and Silver were released in 1999 for the Game Boy Color.
Tokens: 1614

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Question: Which one was the first 3D platformer Mario game?
Answer: The first 3D platformer Mario game is **Super Mario 64**, released in 1996 for the Nintendo

### (Optional) Advanced

In [15]:
# Advanced: persistent long-term memory around the existing state-machine agent.
import importlib

import lib.memory as memory_module
import lib.vector_db as vector_db_module

importlib.reload(vector_db_module)
importlib.reload(memory_module)

from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

long_term_memory = LongTermMemory(
    VectorStoreManager(OPENAI_API_KEY, persist_path="chromadb")
)


def remember_interaction(question: str, answer: str, owner: str) -> None:
    long_term_memory.register(
        MemoryFragment(
            content=f"Question: {question}\nAnswer: {answer}",
            owner=owner,
            namespace="udaplay",
        )
    )


def recall_memories(question: str, owner: str, limit: int = 3) -> list[str]:
    result = long_term_memory.search(
        query_text=question,
        owner=owner,
        namespace="udaplay",
        limit=limit,
    )
    return [fragment.content for fragment in result.fragments]


def invoke_with_long_term_memory(
    question: str,
    session_id: str = "advanced",
):
    memories = recall_memories(question, owner=session_id)
    context = "\n\n".join(memories)
    enriched_question = question
    if context:
        enriched_question = (
            "Relevant remembered context:\n"
            f"{context}\n\nCurrent question: {question}"
        )

    run = agent.invoke(enriched_question, session_id=session_id)
    final_state = run.get_final_state()
    answer = final_state["messages"][-1].content
    remember_interaction(question, answer, owner=session_id)
    return run


remember_interaction(
    "The user prefers concise game research answers.",
    "Preference recorded.",
    owner="advanced-demo",
)
print("Recalled memory:", recall_memories("How should answers be written?", "advanced-demo"))
advanced_run = invoke_with_long_term_memory(
    "When was Pokémon Gold and Silver released?",
    session_id="advanced-demo",
)
print("Advanced answer:", advanced_run.get_final_state()["messages"][-1].content)

Recalled memory: ['Question: The user prefers concise game research answers.\nAnswer: Preference recorded.']
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Advanced answer: **Pokémon Gold and Silver** were released on the following dates:
- **Japan**: November 21, 1999
- **North America**: October 15, 2000
- **Australia**: October 13, 2000
- **Europe**: April 6, 2001

These games were developed for the Game Boy Color and are part of the second generation of the Pokémon series.
